# 第 33 课：VAD 与 Endpoint——何时开始、何时结束

VAD 判断帧是否含语音；endpoint 把一串 VAD 决策变成“开始识别/结束一句”。二者不是同一个模块。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 音频信号前端 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 32 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | VAD、endpoint 状态机、hangover |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：VAD、endpoint 状态机、hangover。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from ipywidgets import interact,FloatSlider,IntSlider
y,sr=sf.read(ROOT/"data"/"spoken_digits_0_to_9_8k.wav");y=y.astype(np.float32);L=int(.025*sr);H=int(.01*sr)
frames=librosa.util.frame(y,frame_length=L,hop_length=H).T;energy=10*np.log10(np.mean(frames**2,axis=1)+1e-12);times=np.arange(len(energy))*H/sr

## 1. 能量 VAD 只是最小基线

In [ ]:
@interact(threshold=FloatSlider(min=-60,max=-15,value=-38,step=1),hangover=IntSlider(min=0,max=30,value=8))
def show_vad(threshold=-38,hangover=8):
    raw=energy>threshold;smooth=raw.copy();left=0
    for i,v in enumerate(raw):
        if v:left=hangover
        elif left>0:smooth[i]=True;left-=1
    fig,ax=plt.subplots(2,1,figsize=(11,5),sharex=True)
    ax[0].plot(np.arange(len(y))/sr,y);ax[0].set_ylabel("Amplitude")
    ax[1].plot(times,energy,label="frame energy");ax[1].axhline(threshold,color="C1");ax[1].fill_between(times,energy.min(),energy.max(),where=smooth,alpha=.2)
    ax[1].set(xlabel="Time (s)",ylabel="dB",title=f"speech frames={smooth.sum()}");plt.show()

## 2. 状态机比逐帧阈值更重要

```text
IDLE --连续若干语音帧--> IN_SPEECH
IN_SPEECH --短静音--> 仍保持
IN_SPEECH --足够长静音--> END
```

起点需要触发帧数，终点需要 silence/hangover；否则键盘声会误触发，词内停顿会过早截断。

In [ ]:
def endpoints(mask,start_trigger=3,end_silence=20):
    state="IDLE";speech_run=silence_run=0;segments=[];start=None
    for i,v in enumerate(mask):
        if state=="IDLE":
            speech_run=speech_run+1 if v else 0
            if speech_run>=start_trigger:start=i-start_trigger+1;state="SPEECH";silence_run=0
        else:
            silence_run=0 if v else silence_run+1
            if silence_run>=end_silence:segments.append((start,i-end_silence+1));state="IDLE";speech_run=0
    if state=="SPEECH":segments.append((start,len(mask)))
    return segments
print(endpoints(energy>-38))

## 3. Endpoint 与最终延迟

终止静音设为 800 ms，就算模型 RTF=0.05，用户通常也至少要等这段静音才得到 final。可以用标点、CTC blank、语义完整性辅助，但误切与延迟仍要权衡。

## 本课测试

1. VAD=false 是否一定代表绝对静音？
2. hangover 的作用是什么？
3. endpoint 为什么直接影响 final latency？
4. 阈值太低会怎样？
5. VAD 是否应该直接删除所有非语音帧再送 CTC？

<details><summary>展开参考答案</summary>

1. 不是，只是判为非语音。2. 防止短暂停顿切断语句。3. 系统等待足够终止静音。4. 噪声误触发、句子难结束。5. 不一定，粗暴删除会破坏时间轴和上下文。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 33 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `VAD`、`endpoint 状态机`、`hangover`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**词内短停顿触发错误 final**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现起点/终点触发并扫描延迟—误切**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把 endpoint 等待加入第 18 课延迟**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：VAD、endpoint 状态机、hangover。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 VAD、endpoint 状态机、hangover。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
